# Step 02 — ALKIS / LoD2 extraction

Assemble the raw ALKIS building dataset from LGLN's LoD2 open data for
Niedersachsen: select the region's tiles, download them, and merge them into
one layer.

| | |
|---|---|
| **Reads** | `data/input/lgln-opengeodata-lod2.geojson`, `data/input/regionalverband_area.gpkg` |
| **Writes** | `02_lod2_region_tiles.gpkg` · `02_alkis_lod2_raw.gpkg` |
| **Needs** | `ogr2ogr` (system binary), `requests` |
| **Runtime** | ~24 min cold, ~23 min warm — the merge always reruns |
| **Disk** | 0.44 GB cached zips + 5.07 GB output |
| **Result** | 1,385,279 buildings as 4,891,343 surface rows |

This is the equivalent of `ALKIS_LOD2-data_extraction.ipynb` in the original
pipeline and does the same job and nothing else: **filter the tile index,
download, merge.**

The original's separate unzip stage is gone — GDAL reads a shapefile straight
out of its `.zip` through `/vsizip/`, which produces a byte-identical merge
(verified on three tiles: same row count, same `gml_id` count, same column set,
same total area, same file size) while never extracting anything. The zips
inflate ~25x, so this avoids roughly **11 GB** of shapefiles and keeps peak disk
at 5.5 GB.

Measured on the full region: 3,620 tiles downloaded in 67 s, merged in
1,383 s, `02_alkis_lod2_raw.gpkg` = 5.07 GB. The original pipeline's equivalent
`merged_all.gpkg` was 5.09 GB. An earlier version of this notebook passed
`-dim XY` and got 4.26 GB in 849 s; section 4 explains why the Z ordinate is
kept after all.

**`02_alkis_lod2_raw.gpkg` is raw.** One row per LoD2 *surface*, not per
building — a tile carries ground, wall and roof surfaces as separate rows
sharing one `gml_id`. Region-wide that is **4,891,343 rows for 1,385,279
buildings, 3.53 surfaces each**. So `gml_id` is **not unique** in this file,
`area_m2` is not computed, and no volume exists yet.
Reducing surfaces to buildings, computing volume, choosing a size threshold,
labelling the `function` codes and deciding whether to merge adjacent polygons
are all later steps' decisions. Section 5 measures the file so those decisions
have numbers to work from.

The download skips what is already on disk, so an interrupted run resumes
rather than restarting.

In [1]:
import os, shutil, sys
from pathlib import Path

# --- locate the pipeline root -------------------------------------------------
# Same reason as step 01: a notebook's working directory is not necessarily its
# own folder, so Path('..') is unreliable. Find the root by its marker file.
def _find_root(start):
    for d in (start, *start.parents):
        if (d / 'config.py').is_file() and (d / 'lib' / 'checks.py').is_file():
            return d
    return None

_nb_dir = Path(globals()['__vsc_ipynb_file__']).parent if '__vsc_ipynb_file__' in globals() else None
ROOT_DIR = _find_root(_nb_dir) if _nb_dir else None
ROOT_DIR = ROOT_DIR or _find_root(Path.cwd())
if ROOT_DIR is None:
    raise RuntimeError(
        'Cannot find the pipeline root (the folder containing config.py). '
        f'Looked upward from notebook dir {_nb_dir} and cwd {Path.cwd()}.'
    )
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# --- point GDAL/PROJ at this env's data files ---------------------------------
# Must run before geopandas is imported. ogr2ogr inherits these too: without
# GDAL_DATA it prints `Cannot find tms_NZTM2000.json` on every single call,
# which across 3,620 tiles buries the real output.
_share = Path(sys.prefix) / 'Library' / 'share'
if not _share.is_dir():
    _share = Path(sys.prefix) / 'share'
if (_share / 'gdal').is_dir():
    os.environ.setdefault('GDAL_DATA', str(_share / 'gdal'))
if (_share / 'proj').is_dir():
    os.environ.setdefault('PROJ_LIB', str(_share / 'proj'))

import subprocess, zipfile, time
from concurrent.futures import ThreadPoolExecutor

# --- find a working ogr2ogr ---------------------------------------------------
# Verified by running it, for the two reasons step 01 verifies osmium: a kernel
# started without `conda activate` has none of this env's binaries on PATH, and
# shutil.which returns the PATHEXT-upper-cased name on Windows.
for _bin in (Path(sys.prefix) / 'Library' / 'bin',
             Path(sys.prefix) / 'Scripts',
             Path(sys.prefix) / 'bin'):
    if _bin.is_dir() and str(_bin) not in os.environ.get('PATH', ''):
        os.environ['PATH'] = str(_bin) + os.pathsep + os.environ.get('PATH', '')


def _find_tool(stem):
    cands = []
    for d in (Path(sys.prefix) / 'Library' / 'bin',
              Path(sys.prefix) / 'Scripts',
              Path(sys.prefix) / 'bin'):
        cands += [d / f'{stem}.exe', d / stem]
    found = shutil.which(stem)
    if found:
        f = Path(found)
        cands += [f.with_suffix(f.suffix.lower()), f]
    for c in cands:
        if not c.is_file():
            continue
        try:
            r = subprocess.run([str(c), '--version'], capture_output=True, text=True)
        except OSError:
            continue
        if r.returncode == 0:
            return str(c), r.stdout.splitlines()[0]
    return None, None


OGR2OGR, _gdal_version = _find_tool('ogr2ogr')
if OGR2OGR is None:
    raise RuntimeError(
        'No working ogr2ogr found. Section 4 needs it: LoD2 tiles are MultiPatch '
        'shapefiles, which shapely cannot parse (`Unknown WKB type 16`), so GDAL '
        'has to do the conversion. Install it with:  '
        'conda install -c conda-forge gdal'
    )

import requests
import pandas as pd
import geopandas as gpd

from config import (
    STUDY_BOUNDARY_FILE, TARGET_CRS, OUTPUT_DIR,
    LOD2_TILE_INDEX_FILE, LOD2_CACHE_DIR, LOD2_ZIP_DIR,
    LOD2_REGION_TILES_FILE, ALKIS_RAW_FILE,
    LOD2_DOWNLOAD_WORKERS, LOD2_DOWNLOAD_TIMEOUT_S, LOD2_DOWNLOAD_RETRIES,
    LOD2_MERGE_GEOM_TYPE, LOD2_MERGE_FLATTEN_Z,
)
from lib.checks import require_file, require_non_empty, require_crs, require_unique

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOD2_ZIP_DIR.mkdir(parents=True, exist_ok=True)

print('Root       :', ROOT_DIR)
print('ogr2ogr    :', _gdal_version, '|', OGR2OGR)
print('Target CRS :', TARGET_CRS)
print('Cache      :', LOD2_CACHE_DIR)

Root       : c:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-FINAL
ogr2ogr    : GDAL 3.13.3 "Iowa City", released 2026/08/13 | c:\Users\Mayur Patel\anaconda3\envs\capacity-final\Library\bin\ogr2ogr.exe
Target CRS : EPSG:25832
Cache      : c:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-FINAL\data\input\lod2_cache


## 1. Input contract

Both inputs are checked before a single tile is fetched — a missing boundary
should fail in a second, not an hour into the downloads.

In [2]:
require_file(LOD2_TILE_INDEX_FILE, 'LoD2 tile index')
require_file(STUDY_BOUNDARY_FILE, 'study boundary')

boundary = gpd.read_file(STUDY_BOUNDARY_FILE)
require_non_empty(boundary, 'boundary')
print(f'  ..  boundary: {boundary.crs.to_string()}, '
      f'{boundary.to_crs(TARGET_CRS).geometry.area.sum() / 1e6:,.0f} km2')

  ok  lgln-opengeodata-lod2.geojson (23.0 MB)
  ok  regionalverband_area.gpkg (0.2 MB)
  ok  boundary: 9 rows
  ..  boundary: EPSG:25832, 5,100 km2


## 2. Select the region's tiles

The index is the statewide catalogue: 37,928 tiles, each a polygon plus a `shp`
URL pointing at a zipped shapefile. It is EPSG:4326 whatever the tiles
themselves use, so the boundary is reprojected to *it* rather than the reverse.

`intersects`, not `within`: a tile the region border cuts through still holds
buildings inside the region. Tiles are 1 km squares, so the overhang is bounded
and harmless — trimming it is a later step's business, if it matters at all.

The selection is written out so the download set is auditable in QGIS, and so a
missing tile can be traced back to a specific square.

In [3]:
tiles_all = gpd.read_file(LOD2_TILE_INDEX_FILE)
require_crs(tiles_all, 'EPSG:4326', 'tile index')
print(f'  ..  statewide tiles: {len(tiles_all):,}')

boundary_4326 = boundary.to_crs(epsg=4326).geometry.union_all()
tiles = tiles_all[tiles_all.geometry.intersects(boundary_4326)].copy().reset_index(drop=True)
require_non_empty(tiles, 'region tiles')

# One row per tile is asserted, not assumed: a duplicated `shp` URL would be
# downloaded twice and merged twice.
require_unique(tiles, 'shp', 'region tiles')

bad = tiles[~tiles['shp'].astype(str).str.startswith('http')]
if len(bad):
    raise AssertionError(f'{len(bad)} region tiles carry no usable `shp` URL')

tiles['tile_name'] = (tiles['shp'].astype(str)
                      .str.rsplit('/', n=1).str[-1]
                      .str.replace(r'\.zip$', '', regex=True))
require_unique(tiles, 'tile_name', 'region tiles')

print(f'  ..  kept {len(tiles):,} of {len(tiles_all):,} '
      f'({100 * len(tiles) / len(tiles_all):.1f} %)')
print(f"  ..  currency (Aktualitaet): {tiles['Aktualitaet'].min()} .. "
      f"{tiles['Aktualitaet'].max()}")

if LOD2_REGION_TILES_FILE.exists():
    LOD2_REGION_TILES_FILE.unlink()
tiles.to_file(LOD2_REGION_TILES_FILE, layer='tiles', driver='GPKG')
print(f'  ok  tile selection -> {LOD2_REGION_TILES_FILE.name}')

  ok  tile index: CRS EPSG:4326
  ..  statewide tiles: 37,928
  ok  region tiles: 3,620 rows
  ok  region tiles.shp: unique and non-null (3,620)
  ok  region tiles.tile_name: unique and non-null (3,620)
  ..  kept 3,620 of 37,928 (9.5 %)
  ..  currency (Aktualitaet): 2024-06-13 00:00:00 .. 2024-07-13 00:00:00
  ok  tile selection -> 02_lod2_region_tiles.gpkg


## 3. Download the tiles

One zipped shapefile per tile from LGLN's object store, `LOD2_DOWNLOAD_WORKERS`
at a time. The original pipeline fetched these one at a time under `tqdm`; the
only change here is concurrency plus the three safeguards below.

* An existing zip is **verified, not assumed** — `testzip()` catches the
  truncated file an interrupted kernel leaves behind, and it is re-fetched.
  Skipping on mere existence, as the original did, silently keeps a corrupt tile.
* A download lands on a `.part` file and is renamed only once complete, so a
  `.zip` on disk is always a whole archive.
* Failures are collected across the whole run and raised only at the end. A
  handful of bad tiles must not discard an hour of good ones — rerun the cell
  and only those retry — but the merge must never start on an incomplete set,
  so the cell does fail once every tile has had its chance.

The merge list is the **selected tiles**, not whatever the cache directory
holds. The cache is shared across regions and can carry zips from an earlier
boundary; globbing it would merge those in silently.

In [4]:
def _zip_ok(path):
    # A complete, readable archive that actually contains a shapefile.
    try:
        with zipfile.ZipFile(path) as z:
            if z.testzip() is not None:
                return False
            return any(n.lower().endswith('.shp') for n in z.namelist())
    except (zipfile.BadZipFile, OSError):
        return False


def _fetch(job):
    url, name = job
    dest = LOD2_ZIP_DIR / f'{name}.zip'
    if dest.exists() and _zip_ok(dest):
        return ('cached', name, None)

    part = dest.with_suffix('.part')
    last = None
    for attempt in range(LOD2_DOWNLOAD_RETRIES):
        try:
            # A zip that exists but failed _zip_ok was truncated by an
            # interrupted run. Inside the try: on Windows a stray handle makes
            # unlink raise PermissionError, which must fail THIS tile only and
            # not escape pool.map and abort the other 3,619.
            dest.unlink(missing_ok=True)
            with requests.get(url, stream=True, timeout=LOD2_DOWNLOAD_TIMEOUT_S) as r:
                r.raise_for_status()
                with open(part, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=1 << 16):
                        if chunk:
                            f.write(chunk)
            if not _zip_ok(part):
                raise OSError('downloaded file is not a readable zip')
            part.replace(dest)  # atomic rename: no half-written .zip is visible
            return ('downloaded', name, None)
        except Exception as e:
            last = e
            time.sleep(1.5 * (attempt + 1))
    part.unlink(missing_ok=True)
    return ('failed', name, f'{type(last).__name__}: {last}')


jobs = list(zip(tiles['shp'].astype(str), tiles['tile_name']))
print(f'Downloading {len(jobs):,} tiles, {LOD2_DOWNLOAD_WORKERS} at a time '
      f'(cold: ~70 s for this region; fully cached: seconds) ...', flush=True)

t0 = time.perf_counter()
results = []
with ThreadPoolExecutor(max_workers=LOD2_DOWNLOAD_WORKERS) as pool:
    for i, res in enumerate(pool.map(_fetch, jobs), 1):
        results.append(res)
        if i % 250 == 0 or i == len(jobs):
            ok = sum(1 for s, _, _ in results if s != 'failed')
            print(f'  ..  {i:>5,}/{len(jobs):,}  ok={ok:,}  '
                  f'[{time.perf_counter() - t0:,.0f}s]', flush=True)

print()
for k, v in pd.Series([s for s, _, _ in results]).value_counts().items():
    print(f'  ..  {k:<11} {v:>6,}')

failed = [(n, e) for s, n, e in results if s == 'failed']
if failed:
    print(f'\n  !!  {len(failed)} tiles failed:')
    for n, e in failed[:10]:
        print(f'        {n}: {e}')
    if len(failed) > 10:
        print(f'        ... and {len(failed) - 10} more')
    raise RuntimeError(
        f'{len(failed)} of {len(jobs):,} tiles could not be downloaded. The '
        'good ones are cached, so rerun this cell and only the failed ones '
        'retry. The merge is not allowed to start on an incomplete set.'
    )

# The merge list is the SELECTION, in tile order, not a glob of the cache
# directory: the cache is shared across regions and may hold zips from an
# earlier boundary, and a glob would merge those in without a word.
zips = [LOD2_ZIP_DIR / f'{n}.zip' for n in tiles['tile_name']]
missing = [z.name for z in zips if not z.is_file()]
if missing:
    raise FileNotFoundError(f'{len(missing)} selected tiles are not in the cache '
                            f'after a clean download pass: {missing[:5]}')
n_extra = len(list(LOD2_ZIP_DIR.glob('*.zip'))) - len(zips)
if n_extra:
    print(f'  ..  {n_extra:,} zips in the cache belong to another selection and '
          'are ignored')
print(f'\n  ok  {len(zips):,} selected tiles cached, '
      f'{sum(p.stat().st_size for p in zips) / 1e9:,.2f} GB')

  ..    250/3,620  ok=250  [2s]
  ..    500/3,620  ok=500  [3s]
  ..    750/3,620  ok=750  [5s]
  ..  1,000/3,620  ok=1,000  [7s]
  ..  1,250/3,620  ok=1,250  [8s]
  ..  1,500/3,620  ok=1,500  [10s]
  ..  1,750/3,620  ok=1,750  [12s]
  ..  2,000/3,620  ok=2,000  [14s]
  ..  2,250/3,620  ok=2,250  [16s]
  ..  2,500/3,620  ok=2,500  [18s]
  ..  2,750/3,620  ok=2,750  [20s]
  ..  3,000/3,620  ok=3,000  [21s]
  ..  3,250/3,620  ok=3,250  [23s]
  ..  3,500/3,620  ok=3,500  [24s]
  ..  3,620/3,620  ok=3,620  [25s]

  ..  cached       3,620

  ok  3,620 selected tiles cached, 0.44 GB


## 4. Merge into one layer

`ogr2ogr -append`, once per tile, into a single `buildings` layer — the same
approach as the original pipeline's final cell, which built `merged_all.gpkg`
that way.

**Why GDAL and not geopandas.** LoD2 shapefiles hold **MultiPatch** geometry —
3D surfaces. `geopandas.read_file` on a raw tile dies with
`GEOSException: ParseException: Unknown WKB type 16`, because shapely has no TIN
type. GDAL is the only thing here that can read them, which is why the merge is
a subprocess and not a `pd.concat`.

**Read straight from the zip.** The source path is `/vsizip/<tile>.zip`, GDAL's
virtual filesystem, so nothing is extracted to disk. Pointing it at the archive
rather than an inner `.shp` also means a tile shipping more than one shapefile
contributes all of them, which is what the original's recursive `*.shp` glob
did. Verified against extract-then-merge on three tiles: identical row count,
`gml_id` count, column set and total area, byte-for-byte the same file size.

One flag the original did not pass: `-nlt MULTIPOLYGONZ`. Appending 3,620 tiles
into one layer needs a single declared geometry type, or a tile whose surfaces
come back as plain `POLYGON` is rejected by a layer created as multi.

**The Z ordinate is kept, and that is not free — it costs ~0.8 GB.** An earlier
version of this notebook passed `-dim XY` to save exactly that, reasoning that
the heights are already in `measHeight`/`Firsthoehe`/`Traufhoehe`/`AbsHoehe`.
That reasoning was wrong. Z does not carry the heights; it carries the **surface
semantics**. It is the only thing that separates a ground surface from a roof
surface — no attribute column does, because every row of a building repeats
identical attribute values.

Measured on a 60-tile sweep: classifying surfaces by Z yields exactly one
`GROUND` surface for **13,833 of 13,833 parts (100.00 %)**, holding across
46.8–824.9 m of terrain, both `DqDach` groups and every surface count. Discard Z
and the only fallback is the original pipeline's "largest area per `gml_id`",
which picks the roof instead of the footprint whenever the roof overhangs —
wrong for **1.15 % of parts, median +32 %, worst +255 %**. Z also makes an exact
volume computable instead of `area × ridge height`, which overstates
pitched-roof buildings by 20–46 %.

The first tile **creates** the layer and every later one appends to it (decided
by whether the output exists, not by loop position, so a failed first tile
cannot turn the other 3,619 into appends to nothing), so a partial merge cannot
be resumed: the output is deleted first and the whole merge
redone, every run. That is deliberate — appending to a layer of unknown state is
how you get silent duplicates — and it is why a warm rerun still costs ~14 min.
Caching protects the downloads, but at 67 s those turned out to be the cheap part:
the merge is 95 % of the runtime.

A tile that fails to merge is reported and the cell **raises** once the loop is
through, so section 5 never measures a file that is missing tiles.

In [5]:
if ALKIS_RAW_FILE.exists():
    try:
        ALKIS_RAW_FILE.unlink()
    except PermissionError as e:
        raise RuntimeError(
            f'{ALKIS_RAW_FILE.name} is locked by another process, so it cannot '
            'be replaced. Anything holding an open SQLite handle does this:\n'
            '  - QGIS, for as long as the layer is loaded (remove the layer or '
            'close the project);\n'
            '  - another Jupyter kernel that read this file and has not been '
            'shut down - including an earlier run of step 03 in this same '
            'session, which is the usual cause;\n'
            '  - a stray python.exe from an interrupted run.\n'
            'Note that a read-only handle is enough to block deletion on '
            'Windows, and probing with open(path) will NOT detect it - only '
            'os.rename does. Close the holder and rerun this cell. '
            f'Original error: {e}'
        ) from None

_base = [OGR2OGR, '-f', 'GPKG', '-nln', 'buildings',
         '-nlt', LOD2_MERGE_GEOM_TYPE, '-t_srs', TARGET_CRS]
if LOD2_MERGE_FLATTEN_Z:
    _base += ['-dim', 'XY']

print(f'Merging {len(zips):,} tiles into {ALKIS_RAW_FILE.name} '
      f'(expect ~14 min; it slows as the GeoPackage grows) ...', flush=True)

t0 = time.perf_counter()
merge_failed = []
for i, z in enumerate(zips, 1):
    # GDAL's virtual filesystem: read the shapefile inside the archive, with
    # nothing extracted. as_posix() because /vsizip/ paths use forward slashes
    # even on Windows.
    src = '/vsizip/' + z.as_posix()
    # Whoever runs first CREATES the layer; everyone after appends. Decided on
    # the file, not on i == 1: if the first tile fails, the next one must still
    # create rather than -append into a datasource that does not exist.
    extra = ['-update', '-append'] if ALKIS_RAW_FILE.exists() else []
    r = subprocess.run(_base + extra + [str(ALKIS_RAW_FILE), src],
                       capture_output=True, text=True)
    if r.returncode != 0:
        merge_failed.append((z.stem, (r.stderr or r.stdout).strip()[:160]))
    if i % 200 == 0 or i == len(zips):
        mb = ALKIS_RAW_FILE.stat().st_size / 1e6 if ALKIS_RAW_FILE.exists() else 0
        print(f'  ..  {i:>5,}/{len(zips):,}  {mb:>9,.0f} MB  '
              f'[{time.perf_counter() - t0:,.0f}s]', flush=True)

if merge_failed:
    print(f'\n  !!  {len(merge_failed)} tiles failed to merge:')
    for n, e in merge_failed[:10]:
        print(f'        {n}: {e}')
    if len(merge_failed) > 10:
        print(f'        ... and {len(merge_failed) - 10} more')
    raise RuntimeError(
        f'{len(merge_failed)} of {len(zips):,} tiles failed to merge, so '
        f'{ALKIS_RAW_FILE.name} is incomplete. Fix the cause (the first error '
        'is printed above) and rerun this cell; it rebuilds from scratch.'
    )

if not ALKIS_RAW_FILE.exists():
    raise RuntimeError('no tile created the layer - was the tile list empty?')

print(f'\n  ok  {ALKIS_RAW_FILE.name}: '
      f'{ALKIS_RAW_FILE.stat().st_size / 1e9:,.2f} GB  '
      f'[{time.perf_counter() - t0:,.0f}s]')

Merging 3,620 tiles into 02_alkis_lod2_raw.gpkg (expect ~14 min; it slows as the GeoPackage grows) ...
  ..    200/3,620        270 MB  [48s]
  ..    400/3,620        462 MB  [94s]
  ..    600/3,620        756 MB  [143s]
  ..    800/3,620      1,021 MB  [193s]
  ..  1,000/3,620      1,278 MB  [237s]
  ..  1,200/3,620      1,558 MB  [287s]
  ..  1,400/3,620      1,834 MB  [338s]
  ..  1,600/3,620      2,140 MB  [391s]
  ..  1,800/3,620      2,407 MB  [441s]
  ..  2,000/3,620      2,698 MB  [492s]
  ..  2,200/3,620      2,933 MB  [558s]
  ..  2,400/3,620      3,275 MB  [641s]
  ..  2,600/3,620      3,622 MB  [727s]
  ..  2,800/3,620      3,854 MB  [811s]
  ..  3,000/3,620      4,135 MB  [891s]
  ..  3,200/3,620      4,464 MB  [960s]
  ..  3,400/3,620      4,750 MB  [1,016s]
  ..  3,600/3,620      5,034 MB  [1,072s]
  ..  3,620/3,620      5,075 MB  [1,081s]

  ok  02_alkis_lod2_raw.gpkg: 5.07 GB  [1,081s]


## 5. Verify and measure

The merged layer is read back through GDAL's own metadata rather than loaded
into memory — it is a multi-gigabyte file and a full read would be pointless
here.

What the numbers mean, and why `gml_id` is deliberately not checked for
uniqueness: a LoD2 tile stores one row **per surface class**, all sharing a
`gml_id` and repeating the same attributes. For `DENILD61000068Eu` there are
three rows — the two roof planes (186.9 m² projected), the six walls (0.0 m²
once flattened, being vertical), and the ground surface (186.9 m², the actual
footprint).

Region-wide that comes to **4,891,343 surface rows for 1,385,279 buildings,
3.53 each**, median 4 and max 4 — but **min 1**, so a minority of buildings
arrive as a single surface. Any reduction rule has to handle that case rather
than assume a ground surface is always present.

Reducing these surfaces to one row per building is the first thing the next step
has to decide. Rerunning this cell after a re-download is also how you check
whether the region's structure has changed under you.

### What has been verified independently

Two checks were run outside this notebook, both deliberately using a different
code path from the one that produced the file:

**Merge fidelity.** Every source zip's feature count was read with `pyogrio`
directly and summed, then compared against the merged layer. The merge itself
ran through `ogr2ogr` subprocesses, so the two numbers come from independent
readers:

```
features in the 3,620 source zips : 4,891,343
rows in 02_alkis_lod2_raw.gpkg    : 4,891,343
difference                        : +0
unreadable zips                   : 0
zips with more than one layer     : 0
```

No feature lost, none duplicated, and no zip carried a second layer that the
merge would have ignored in silence.

**Selection completeness.** Row-count conservation only proves the merge is
faithful to the tiles it was handed; it cannot see a tile that was never
selected. Unioning the tile footprints and subtracting the study boundary gives
`0.0000 km2` attributable to the selection - 3,620 statewide tiles intersect the
region and all 3,620 were taken.

### The region is only ~68 % covered by LoD2, and that is normal

The same check turned up something that looks alarming and is not:

```
region                                 5,094.1 km2
covered by the selected tiles          3,576.1 km2
not covered by ANY statewide tile      1,627.3 km2   (31.9 %)
```

LGLN publishes a tile only where there is something to model, so a 1 km square
of Harz forest, farmland or water never enters the statewide index at all - the
index covers 38,340 km2 of Niedersachsen's ~47,700. This is a property of the
source, not of the selection, and no change here can recover it.

That it is genuinely empty land was confirmed against OSM, which is independent
of LGLN entirely. The exact figure depends on when you call a building "in the
gap", so both measures are given:

```
                             buildings in gap    share    density ratio
centroid inside the gap                 3,682   0.72 %          1 : 64
footprint touches the gap                8,678   1.70 %          1 : 27
```

Either way a third of the region's *area* holds one to two percent of its
buildings. The 4,996-building difference between the two rows is footprints that
straddle a tile edge - which is itself consistent with the anomalies below
clustering on the index boundary rather than deep in the gap.

POIs in the gap are a median **743 m** from the nearest ALKIS building, i.e.
genuinely in fields and forest (2,106 by the touches measure, 1,656 by
centroid).

One caveat found while checking this: 1,656 POIs (4.9 %) fall in the gap,
mostly signposts, boundary stones, memorials and ruins that need no building.
Of the 30 carrying commercial tags, the median distance to an ALKIS building we
hold is **0.0 m** and 22 of 30 are within 25 m - they sit at tile-index edges
rather than outside the data. The handful genuinely far from any building are
things like `Weihnachtsbaum Verkauf Schmidt` at 766 m, a Christmas-tree field,
which correctly has no building. Assigning POIs with no building underneath them
is step 04's problem, not this one's.

In [6]:
import pyogrio

info = pyogrio.read_info(ALKIS_RAW_FILE, layer='buildings')
print(f'  ..  layer      : buildings')
print(f'  ..  rows       : {info["features"]:,}')
print(f'  ..  geometry   : {info["geometry_type"]}')
print(f'  ..  CRS        : {info["crs"]}')
print(f'  ..  fields     : {len(info["fields"])}')
print(f'  ..  bounds     : {tuple(round(v) for v in info["total_bounds"])}')
print()
print('  ..  columns:', ', '.join(info['fields']))

# gml_id alone, no geometry: cheap enough on a file this size and it is the one
# number the next step needs.
ids = pyogrio.read_dataframe(ALKIS_RAW_FILE, layer='buildings',
                             columns=['gml_id'], read_geometry=False)
n_bld = ids['gml_id'].nunique()
print()
print(f'  ..  surface rows        : {len(ids):,}')
print(f'  ..  distinct gml_id     : {n_bld:,}')
print(f'  ..  surfaces / building : {len(ids) / n_bld:.2f}')
print(f'  ..  rows with no gml_id : {int(ids["gml_id"].isna().sum()):,}')

per = ids['gml_id'].value_counts()
print(f'  ..  surfaces per building: min {per.min()}, median {per.median():.0f}, '
      f'max {per.max():,}')

require_non_empty(ids, 'merged surfaces')
print(f'\n  ok  extraction complete: {n_bld:,} buildings as {len(ids):,} surface rows')

  ..  layer      : buildings
  ..  rows       : 4,891,343
  ..  geometry   : MultiPolygon Z
  ..  CRS        : EPSG:25832
  ..  fields     : 27
  ..  bounds     : (567528, 5722464, 641757, 5853020)

  ..  columns: measHeight, gml_id, externRef, function, DqDach, DqLage, DqBoden, creationDa, GrundrissA, LetzteAend, Geom2DRef, roofType, AGS, Land, Stadt, Strasse, HausNr, Name, DachNeig, DachOri, DachFlaech, Firsthoehe, Traufhoehe, AbsHoehe, DachName, Eigentum, Lizenz

  ..  surface rows        : 4,891,343
  ..  distinct gml_id     : 1,385,279
  ..  surfaces / building : 3.53
  ..  rows with no gml_id : 0
  ..  surfaces per building: min 1, median 4, max 4
  ok  merged surfaces: 4,891,343 rows

  ok  extraction complete: 1,385,279 buildings as 4,891,343 surface rows


## 6. QGIS checkpoint

Load `02_alkis_lod2_raw.gpkg` and `02_lod2_region_tiles.gpkg` alongside
`regionalverband_area.gpkg`, then confirm:

1. **Coverage** — do NOT expect the buildings to blanket the region. About
   **31.9 % of the area has no LoD2 tile at all**, because LGLN publishes a tile
   only where there is something to model; the Harz, farmland and water are
   simply absent from the statewide index. Verified against OSM: that 31.9 % of
   area holds 0.72 % of the region's buildings.

   So a hole is only a bug if it is **square, 1 km, and inside built-up area**.
   Load `02_lod2_region_tiles.gpkg` to see which squares should have data: a
   hole with no tile behind it is expected, a hole *with* a tile behind it is a
   download or merge failure, and the section 3 and 4 reports name it.
2. **Alignment** — footprints sit on a basemap correctly and line up with
   `01_all_buildings_osm.gpkg`. Everything in the Atlantic near (0, 0) means a
   CRS was assigned rather than reprojected.
3. **Overlapping surfaces are expected** — clicking one building returns
   several features with the same `gml_id`. That is the raw structure, not an
   error. Filter `"gml_id" = '<some id>'` and you should see the ground surface
   and the roof planes stacked on each other, plus wall rows with no visible
   area.

Note that this layer is not the deliverable — it is the source for the next
step, which reduces surfaces to buildings and computes volume.